
# MỨC 4: RANDOM FOREST – HOLD-OUT

## 1. Giới thiệu

**Đề tài:** Dự đoán giá kim cương bằng thuật toán Random Forest.

**Dữ liệu:** Diamonds Prices2022.csv.

**Phương pháp:** Hold-out 80% Train – 20% Test.

**Mục tiêu:** Mở rộng phạm vi tìm kiếm siêu tham số Random Forest để đánh giá khả năng cải thiện hiệu suất dự đoán.

Quy trình thực nghiệm:

1. Đọc và tiền xử lý dữ liệu giống mức 2 và mức 3.
2. Chia dữ liệu thành 80% Train và 20% Test.
3. Xây dựng Pipeline mã hóa dữ liệu và Random Forest.
4. Sử dụng GridSearchCV với 3-Fold trên Train.
5. Chọn cấu hình có RMSE CV thấp nhất.
6. Đánh giá mô hình tốt nhất trên tập Test.

Tập Test không tham gia quá trình lựa chọn siêu tham số.


Đọc dữ liệu

In [1]:

# CELL 2: IMPORT CAC THU VIEN CAN THIET

import numpy as np  # Thu vien tinh toan so hoc

import pandas as pd  # Thu vien doc va xu ly du lieu

from pathlib import Path  # Thu vien xu ly duong dan tep

from sklearn.model_selection import train_test_split  # Chia Train/Test

from sklearn.model_selection import GridSearchCV  # Tim kiem sieu tham so

from sklearn.model_selection import KFold  # Chia fold trong GridSearch

from sklearn.preprocessing import OrdinalEncoder  # Ma hoa thuoc tinh phan loai

from sklearn.compose import ColumnTransformer  # Ap dung ma hoa cho cac cot

from sklearn.pipeline import Pipeline  # Ket hop tien xu ly va mo hinh

from sklearn.ensemble import RandomForestRegressor  # Mo hinh Random Forest

from sklearn.metrics import mean_squared_error  # Tinh MSE

from sklearn.metrics import mean_absolute_error  # Tinh MAE

from sklearn.metrics import r2_score  # Tinh R2

print("Da import thu vien thanh cong!")


# CELL 4: DOC DU LIEU GOC

# Duong dan khi notebook chay tu experiments/random_forest
file_path = Path("../../data/raw/Diamonds Prices2022.csv")

# Duong dan du phong khi kernel chay tu thu muc goc du an
if not file_path.exists():
    file_path = Path("data/raw/Diamonds Prices2022.csv")

# Bao loi neu khong tim thay tep CSV
if not file_path.exists():
    raise FileNotFoundError("Khong tim thay file Diamonds Prices2022.csv")

# Doc CSV va su dung cot dau tien lam chi so dong
diamonds = pd.read_csv(file_path, index_col=0)

# In duong dan tep dang su dung
print("Duong dan:", file_path.resolve())

# In kich thuoc du lieu ban dau
print("Kich thuoc ban dau:", diamonds.shape)

# Dem tong so gia tri thieu
print("So gia tri thieu:", diamonds.isnull().sum().sum())

# Dem so dong trung lap
print("So dong trung lap:", diamonds.duplicated().sum())

# Xac dinh cac dong co kich thuoc khong hop le
invalid_size = (
    (diamonds["x"] <= 0) |
    (diamonds["y"] <= 0) |
    (diamonds["z"] <= 0)
)

# In so dong co kich thuoc khong hop le
print("So dong kich thuoc khong hop le:", invalid_size.sum())

# Hien thi 5 dong dau tien
display(diamonds.head())


Da import thu vien thanh cong!
Duong dan: H:\My Drive\Ganka là phù thủy á nha\máy học ứng dụng\project\diamond_prediction\data\raw\Diamonds Prices2022.csv
Kich thuoc ban dau: (53943, 10)
So gia tri thieu: 0
So dong trung lap: 149
So dong kich thuoc khong hop le: 20


,carat,cut,color,clarity,depth,table,price,x,y,z
1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


Tiền xử lý dữ liệu

In [2]:

# CELL 6: TIEN XU LY DU LIEU

# Tao ban sao cua du lieu goc
df = diamonds.copy()

# Luu so dong ban dau
initial_rows = len(df)

# Loai bo cac dong co gia tri thieu
df = df.dropna()

# Luu so dong sau khi xu ly gia tri thieu
after_missing = len(df)

# Chi giu cac dong co kich thuoc lon hon 0
df = df[
    (df["x"] > 0) &
    (df["y"] > 0) &
    (df["z"] > 0)
]

# Luu so dong sau khi loc kich thuoc
after_dimensions = len(df)

# Xoa cac dong trung lap
df = df.drop_duplicates()

# Dat lai chi so dong tu 0
df = df.reset_index(drop=True)

# In so dong bi loai do thieu du lieu
print("Dong thieu bi loai:", initial_rows - after_missing)

# In so dong bi loai do kich thuoc khong hop le
print("Dong kich thuoc khong hop le bi loai:",
      after_missing - after_dimensions)

# In so dong trung lap bi loai
print("Dong trung lap bi loai:",
      after_dimensions - len(df))

# In so dong con lai
print("So dong sau tien xu ly:", len(df))

# Kiem tra so gia tri thieu con lai
print("Gia tri thieu con lai:", df.isnull().sum().sum())

# Kiem tra so dong trung lap con lai
print("Dong trung lap con lai:", df.duplicated().sum())

# Hien thi 5 dong dau tien
display(df.head())


Dong thieu bi loai: 0
Dong kich thuoc khong hop le bi loai: 20
Dong trung lap bi loai: 148
So dong sau tien xu ly: 53775
Gia tri thieu con lai: 0
Dong trung lap con lai: 0


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


Chia tập dữ liệu

In [3]:

# CELL 8: CHIA DU LIEU HOLD-OUT

# Lay tat ca cac cot tru price lam dac trung dau vao
X = df.drop(columns=["price"])

# Lay cot price lam bien muc tieu
y = df["price"]

# Chia du lieu thanh Train va Test
X_train, X_test, y_train, y_test = train_test_split(
    X,  # Cac thuoc tinh dau vao
    y,  # Gia kim cuong can du doan
    test_size=0.2,  # 20% du lieu dung lam Test
    random_state=42  # Co dinh cach chia
)

# In tong so mau
print("Tong so mau:", len(df))

# In so mau Train
print("So mau Train:", len(X_train))

# In so mau Test
print("So mau Test:", len(X_test))


Tong so mau: 53775
So mau Train: 43020
So mau Test: 10755



## 5. Xây dựng Pipeline

Sử dụng OrdinalEncoder để mã hóa ba thuộc tính:
- color
- clarity
- cut

Kết hợp bộ mã hóa và Random Forest trong Pipeline.

Pipeline đảm bảo bộ mã hóa được fit trên phần Train của từng fold trong GridSearchCV, tránh rò rỉ dữ liệu.

Các thuộc tính số được giữ nguyên.


In [4]:

# CELL 10: TAO PIPELINE RANDOM FOREST

# Thu tu chat luong mau sac tu thap den cao
color_order = ["J", "I", "H", "G", "F", "E", "D"]

# Thu tu do tinh khiet tu thap den cao
clarity_order = [
    "I1", "SI2", "SI1", "VS2",
    "VS1", "VVS2", "VVS1", "IF"
]

# Thu tu chat luong giac cat tu thap den cao
cut_order = [
    "Fair", "Good", "Very Good",
    "Premium", "Ideal"
]

# Danh sach cac cot can ma hoa
categorical_cols = ["color", "clarity", "cut"]

# Khoi tao bo ma hoa voi thu tu co dinh
encoder = OrdinalEncoder(
    categories=[
        color_order,  # Thu tu cua color
        clarity_order,  # Thu tu cua clarity
        cut_order  # Thu tu cua cut
    ]
)

# Tao bo tien xu ly du lieu
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", encoder, categorical_cols)  # Ma hoa 3 cot
    ],
    remainder="passthrough"  # Giu nguyen cac cot so
)

# Khoi tao mo hinh Random Forest
rf_model = RandomForestRegressor(
    random_state=42,  # Co dinh tinh ngau nhien
    n_jobs=-1  # Su dung cac loi CPU kha dung
)

# Ket hop bo ma hoa va Random Forest
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),  # Buoc ma hoa
        ("model", rf_model)  # Buoc huan luyen
    ]
)

# Thong bao
print("Da tao Pipeline thanh cong!")


Da tao Pipeline thanh cong!



## 6. Tìm kiếm siêu tham số bằng GridSearchCV

Ở mức 4, nhóm mở rộng phạm vi tìm kiếm so với mức 3.

Các siêu tham số:

- n_estimators: [100, 150, 200]
- max_depth: [None, 20, 30]
- min_samples_leaf: [1, 2]
- max_features: [0.8, 1.0]

Tổng cộng 36 cấu hình.

Mỗi cấu hình được đánh giá bằng 3-Fold Cross-validation trên tập Train.

Tiêu chí lựa chọn là RMSE trung bình thấp nhất.


In [5]:

# CELL 12: TIM KIEM SIEU THAM SO MUC 4
import time  # Thu vien tinh thoi gian
# Khai bao pham vi tim kiem sieu tham so
param_grid = {
    "model__n_estimators": [100, 150, 200],  # So cay trong rung
    "model__max_depth": [None, 20, 30],  # Do sau toi da
    "model__min_samples_leaf": [1, 2],  # So mau toi thieu tai nut la
    "model__max_features": [0.8, 1.0]  # Ti le dac trung moi lan chia
}

# Khoi tao bo chia 3-Fold tren tap Train
cv = KFold(
    n_splits=3,  # Chia Train thanh 3 fold
    shuffle=True,  # Tron du lieu truoc khi chia
    random_state=42  # Co dinh cach chia
)

# Tinh so cau hinh duoc thu
number_of_configs = (
    len(param_grid["model__n_estimators"])
    * len(param_grid["model__max_depth"])
    * len(param_grid["model__min_samples_leaf"])
    * len(param_grid["model__max_features"])
)

# In so cau hinh
print("So cau hinh:", number_of_configs)

# In tong so luot huan luyen CV
print("So luot CV:", number_of_configs * cv.get_n_splits())

# Khoi tao GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline,  # Pipeline can toi uu
    param_grid=param_grid,  # Pham vi sieu tham so
    scoring="neg_root_mean_squared_error",  # Chon theo RMSE
    cv=cv,  # Su dung 3-Fold
    n_jobs=1,  # Tranh song song long nhau
    refit=True,  # Huan luyen lai mo hinh tot nhat
    verbose=1,  # In tien do
    return_train_score=True  # Luu diem Train
)

# Chay tim kiem tren tap Train
# Bắt đầu tính thời gian GridSearchCV
start_time = time.perf_counter()

# Tìm kiếm siêu tham số và huấn luyện
grid_search.fit(X_train, y_train)

# Tính thời gian
training_time = time.perf_counter() - start_time

print(f"Thời gian GridSearchCV: {training_time:.4f} giây")

# In cau hinh sieu tham so tot nhat
print("Sieu tham so tot nhat:")
print(grid_search.best_params_)

# In RMSE trung binh CV tot nhat
print(f"RMSE CV tot nhat: {-grid_search.best_score_:.4f}")


So cau hinh: 36
So luot CV: 108
Fitting 3 folds for each of 36 candidates, totalling 108 fits
Thời gian GridSearchCV: 296.4895 giây
Sieu tham so tot nhat:
{'model__max_depth': 30, 'model__max_features': 0.8, 'model__min_samples_leaf': 2, 'model__n_estimators': 200}
RMSE CV tot nhat: 552.4976


In [6]:

# CELL 14: HIEN THI KET QUA GRID SEARCH

# Chuyen ket qua GridSearchCV thanh DataFrame
cv_results = pd.DataFrame(grid_search.cv_results_)

# Chon cac cot can thiet
search_results = cv_results[
    [
        "param_model__n_estimators",  # So cay
        "param_model__max_depth",  # Do sau toi da
        "param_model__min_samples_leaf",  # So mau tai nut la
        "param_model__max_features",  # Ti le dac trung
        "mean_test_score",  # Diem CV trung binh
        "std_test_score",  # Do lech chuan diem CV
        "rank_test_score"  # Thu hang
    ]
].copy()

# Doi diem am thanh RMSE duong
search_results["RMSE_CV"] = -search_results["mean_test_score"]

# Lay do lech chuan cua RMSE
search_results["RMSE_STD"] = search_results["std_test_score"]

# Sap xep theo RMSE tu thap den cao
search_results = search_results.sort_values("RMSE_CV")

# Hien thi 10 cau hinh co RMSE thap nhat
display(search_results.head(10).round(4))


,param_model__n_estimators,param_model__max_depth,param_model__min_samples_leaf,param_model__max_features,mean_test_score,std_test_score,rank_test_score,RMSE_CV,RMSE_STD
29,200,30,2,0.8,-552.4976,2.3448,1,552.4976,2.3448
5,200,None,2,0.8,-552.4976,2.3995,2,552.4976,2.3995
17,200,20,2,0.8,-552.7010,2.2894,3,552.7010,2.2894
16,150,20,2,0.8,-552.8661,2.0546,4,552.8661,2.0546
13,150,20,1,0.8,-552.9377,2.5743,5,552.9377,2.5743
14,200,20,1,0.8,-552.9768,2.2888,6,552.9768,2.2888
4,150,None,2,0.8,-553.2144,2.3273,7,553.2144,2.3273
28,150,30,2,0.8,-553.2515,2.3263,8,553.2515,2.3263
26,200,30,1,0.8,-554.0979,2.5188,9,554.0979,2.5188
15,100,20,2,0.8,-554.1056,1.3377,10,554.1056,1.3377



## 8. Đánh giá mô hình tốt nhất trên tập Test

Lấy mô hình tốt nhất từ GridSearchCV.

Mô hình đã được huấn luyện lại trên toàn bộ tập Train.

Sử dụng mô hình để dự đoán giá kim cương trên tập Test.

Tính các chỉ số MSE, RMSE, MAE và R².

Không sử dụng tập Test để điều chỉnh siêu tham số.


In [7]:

# CELL 14: HIEN THI KET QUA GRID SEARCH

# Chuyen ket qua GridSearchCV thanh DataFrame
cv_results = pd.DataFrame(grid_search.cv_results_)

# Chon cac cot can thiet
search_results = cv_results[
    [
        "param_model__n_estimators",  # So cay
        "param_model__max_depth",  # Do sau toi da
        "param_model__min_samples_leaf",  # So mau tai nut la
        "param_model__max_features",  # Ti le dac trung
        "mean_test_score",  # Diem CV trung binh
        "std_test_score",  # Do lech chuan diem CV
        "rank_test_score"  # Thu hang
    ]
].copy()

# Doi diem am thanh RMSE duong
search_results["RMSE_CV"] = -search_results["mean_test_score"]

# Lay do lech chuan cua RMSE
search_results["RMSE_STD"] = search_results["std_test_score"]

# Sap xep theo RMSE tu thap den cao
search_results = search_results.sort_values("RMSE_CV")

# Hien thi 10 cau hinh co RMSE thap nhat
display(search_results.head(10).round(4))


,param_model__n_estimators,param_model__max_depth,param_model__min_samples_leaf,param_model__max_features,mean_test_score,std_test_score,rank_test_score,RMSE_CV,RMSE_STD
29,200,30,2,0.8,-552.4976,2.3448,1,552.4976,2.3448
5,200,None,2,0.8,-552.4976,2.3995,2,552.4976,2.3995
17,200,20,2,0.8,-552.7010,2.2894,3,552.7010,2.2894
16,150,20,2,0.8,-552.8661,2.0546,4,552.8661,2.0546
13,150,20,1,0.8,-552.9377,2.5743,5,552.9377,2.5743
14,200,20,1,0.8,-552.9768,2.2888,6,552.9768,2.2888
4,150,None,2,0.8,-553.2144,2.3273,7,553.2144,2.3273
28,150,30,2,0.8,-553.2515,2.3263,8,553.2515,2.3263
26,200,30,1,0.8,-554.0979,2.5188,9,554.0979,2.5188
15,100,20,2,0.8,-554.1056,1.3377,10,554.1056,1.3377


In [8]:

# Lay mo hinh tot nhat da huan luyen
best_model = grid_search.best_estimator_

# Du doan gia kim cuong tren tap Test
y_pred = best_model.predict(X_test)

# Tinh cac chi so danh gia
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# In ket qua
print("KET QUA MUC 4 - HOLD-OUT")
print(f"MSE Test : {mse:.4f}")
print(f"RMSE Test: {rmse:.4f}")
print(f"MAE Test : {mae:.4f}")
print(f"R2 Test  : {r2:.4f}")


KET QUA MUC 4 - HOLD-OUT
MSE Test : 267836.5098
RMSE Test: 517.5292
MAE Test : 263.2254
R2 Test  : 0.9831
